# Load Tether Analysis XArray Data

This notebook demonstrates how to load and work with tether analysis data exported in XArray format from the PyFMGUI DyNaMo tether analysis tool.

The notebook covers:
1. Loading session data from NetCDF files
2. Loading analysis results from NetCDF files 
3. Basic data exploration and visualization
4. Working with the structured scientific data format

In [1]:
# Import required libraries
# import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from plotly.subplots import make_subplots
import plotly.graph_objects as go

import plotly.express as px



In [2]:
# open csv file as dataframe
csv_file_path = '/Users/evillz/Data/article/2025_07_01_THP1_phd/sessions/batch_analysis_results/detailed_plateau_data.csv'
csv_file_path = '/Users/evillz/Data/article/2025_07_01_THP1_phd/sessions/velocity_normalized_thp1_cell1/batch_analysis_results/detailed_plateau_data.csv'
csv_file_path = '/Users/evillz/Data/article/final_yey/batch_analysis_results/detailed_plateau_data.csv'
csv_file_path = '/Users/evillz/Data/article/final_yey/final/yess/batch_analysis_results/detailed_plateau_data.csv'

df = pd.read_csv(csv_file_path)


In [3]:
# Define the list of tether session CSV files to concatenate
# You can either specify individual files or use glob to find files matching a pattern

import os
import glob

# Method 1: Specify individual session files
tether_session_files = [
    '/Users/evillz/Data/article/2025_07_02_THP1_phd/final/tether_session_20250807_135516_final.csv',
    '/Users/evillz/Data/article/20205_07_22_Thp1_AC10/canti1/cell1/final/tether_session_20250807_135014_final.csv',
    '/Users/evillz/Data/article/2025_07_01_THP1_phd/sessions/velocity_normalized_thp1_cell1/final/tether_session_20250807_113419.csv',
    # Add more session files here as needed
    # '/path/to/session_file_2.csv',
    # '/path/to/session_file_3.csv',
]

# Method 2: Alternatively, use glob to find all session files in a directory
# session_directory = '/Users/evillz/Data/article/2025_07_02_THP1_phd/sessions'
# tether_session_files = glob.glob(os.path.join(session_directory, '**/tether_session_*.csv'), recursive=True)

print(f"Found {len(tether_session_files)} session files to concatenate:")
for file in tether_session_files:
    print(f"  - {os.path.basename(file)}")

# Initialize list to store dataframes
tether_dataframes = []

# Process each session file
for file in tether_session_files:
    print(f"\nProcessing: {os.path.basename(file)}")
    
    try:
        # Read the CSV file
        df_session = pd.read_csv(file)
        
        # Check if this is a metadata row (first row often contains session metadata)
        if len(df_session) > 1 and df_session.iloc[0].get('filename', '').startswith('METADATA'):
            print(f"  Skipping metadata row in {os.path.basename(file)}")
            df_session = df_session.iloc[1:].reset_index(drop=True)
        
        # Add source file information for tracking
        df_session['source_session_file'] = os.path.basename(file)
        
        print(f"  Loaded {len(df_session)} rows from {os.path.basename(file)}")
        tether_dataframes.append(df_session)
        
    except Exception as e:
        print(f"  Error loading {file}: {e}")

# Concatenate all dataframes
if tether_dataframes:
    concatenated_tether_data = pd.concat(tether_dataframes, ignore_index=True)
    print(f"\nSuccessfully concatenated {len(tether_dataframes)} session files")
    print(f"Total rows in concatenated data: {len(concatenated_tether_data)}")
    # print(f"Total unique files: {concatenated_tether_data['filename'].nunique()}")
    
    # Display basic info about the concatenated data
    print(f"\nConcatenated data shape: {concatenated_tether_data.shape}")
    print(f"Columns: {list(concatenated_tether_data.columns)}")
else:
    print("No session files were successfully loaded!")
    concatenated_tether_data = pd.DataFrame()

# save concatenated data to a CSV file
output_file = '/Users/evillz/Data/article/2025_07_02_THP1_phd/final/concatenated_tether_data.csv'
concatenated_tether_data.to_csv(output_file, index=False)
print(f"Saved concatenated data to {output_file}")

Found 3 session files to concatenate:
  - tether_session_20250807_135516_final.csv
  - tether_session_20250807_135014_final.csv
  - tether_session_20250807_113419.csv

Processing: tether_session_20250807_135516_final.csv
  Error loading /Users/evillz/Data/article/2025_07_02_THP1_phd/final/tether_session_20250807_135516_final.csv: [Errno 2] No such file or directory: '/Users/evillz/Data/article/2025_07_02_THP1_phd/final/tether_session_20250807_135516_final.csv'

Processing: tether_session_20250807_135014_final.csv
  Loaded 910 rows from tether_session_20250807_135014_final.csv

Processing: tether_session_20250807_113419.csv
  Loaded 245 rows from tether_session_20250807_113419.csv

Successfully concatenated 2 session files
Total rows in concatenated data: 1155

Concatenated data shape: (1155, 9)
Columns: ['local_file_path', 'file_name', 'bool_good_curve', 'analysis_status', 'calc_ret_vel', 'file_parameters', 'plateau_selections', 'current_file_index', 'source_session_file']


OSError: Cannot save file into a non-existent directory: '/Users/evillz/Data/article/2025_07_02_THP1_phd/final'